# Assignment 09: Transformers and MLflow: Downstream Tasks and Experiment Tracking

|                |                       |
| -------------- | --------------------- |
| **Name**       | Vishani Raveendran    |
| **IIT Number** | 20260276              |
| **Course**     | AI Tools and Concepts |

---


## Setup

Load the shared libraries and point MLflow at a local SQLite tracking store (`./mlflow.db`)
inside this assignment folder, so `mlflow ui` can be launched from here to inspect every run.


In [13]:
import os
import pandas as pd
import mlflow
from transformers import pipeline, set_seed

mlflow.set_tracking_uri("sqlite:///" + os.path.abspath("mlflow.db"))
print("MLflow tracking URI:", mlflow.get_tracking_uri())

MLflow tracking URI: sqlite:////Users/vishaniraveendran/Library/CloudStorage/GoogleDrive-vishani.20260276@iit.ac.lk/My Drive/MSC-Vish/First Year/Ai Tools and Concepts/Assignment/Assignements/A09/mlflow.db


## Task 1: Sentiment Classification

Use the pretrained `distilbert-base-uncased-finetuned-sst-2-english` model (via the Hugging Face
`pipeline` API) to classify custom sentences as `POSITIVE` / `NEGATIVE`. Each batch of sentences
is self-labeled with an expected sentiment so accuracy can be computed, then every run is tracked
in MLflow: model name, sample count, accuracy, and the predictions CSV as an artifact. Two runs
are logged — a straightforward batch and a harder batch with negation/sarcasm — to compare how
accuracy shifts with sentence difficulty.


In [14]:
sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=-1,
)


def run_sentiment_experiment(run_name, sentences, true_labels):
    predictions = sentiment_pipe(sentences)
    pred_labels = [p["label"] for p in predictions]
    pred_scores = [p["score"] for p in predictions]
    accuracy = sum(p == t for p, t in zip(pred_labels, true_labels)) / len(true_labels)

    df = pd.DataFrame({
        "sentence": sentences,
        "true_label": true_labels,
        "predicted_label": pred_labels,
        "confidence": pred_scores,
    })

    artifact_path = f"{run_name}_predictions.csv"
    df.to_csv(artifact_path, index=False)

    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("model_name", "distilbert-base-uncased-finetuned-sst-2-english")
        mlflow.log_param("num_samples", len(sentences))
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_artifact(artifact_path)

    os.remove(artifact_path)
    print(f"[{run_name}] accuracy = {accuracy:.2%}")
    return df


mlflow.set_experiment("A09_Task1_Sentiment_Classification")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 7223.65it/s]


<Experiment: artifact_location=('/Users/vishaniraveendran/Library/CloudStorage/GoogleDrive-vishani.20260276@iit.ac.lk/My '
 'Drive/MSC-Vish/First Year/Ai Tools and '
 'Concepts/Assignment/Assignements/A09/mlruns/1'), creation_time=1785590050487, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1785590050487, lifecycle_stage='active', name='A09_Task1_Sentiment_Classification', tags={}, trace_location=None, workspace='default'>

In [15]:
# Run 1: clear-cut sentences (10 positive, 10 negative)
sentences_run1 = [
    "The movie was absolutely fantastic and kept me on the edge of my seat.",
    "I loved the customer service at this restaurant.",
    "This is the best phone I have ever owned.",
    "The concert last night was an unforgettable experience.",
    "Her presentation was clear, engaging, and well organized.",
    "I am so happy with how my new apartment turned out.",
    "The team did an amazing job finishing the project early.",
    "What a wonderful sunny day for a picnic.",
    "The book was a delightful and heartwarming read.",
    "Our vacation in Bali exceeded all of our expectations.",
    "The food was cold and the service was incredibly slow.",
    "I regret buying this laptop, it crashes constantly.",
    "The traffic this morning was a nightmare.",
    "This is the worst customer support I have ever dealt with.",
    "The hotel room smelled awful and was not cleaned properly.",
    "I was extremely disappointed with the ending of the film.",
    "The product broke after just two days of use.",
    "Waiting in line for three hours was frustrating and exhausting.",
    "The lecture was boring and difficult to follow.",
    "I can't believe how rude the staff were to us.",
]
labels_run1 = ["POSITIVE"] * 10 + ["NEGATIVE"] * 10

df_run1 = run_sentiment_experiment("distilbert_run1_general", sentences_run1, labels_run1)
df_run1

[distilbert_run1_general] accuracy = 100.00%


,sentence,true_label,predicted_label,confidence
0,The movie was absolutely fantastic and kept me...,POSITIVE,POSITIVE,0.999883
1,I loved the customer service at this restaurant.,POSITIVE,POSITIVE,0.999818
2,This is the best phone I have ever owned.,POSITIVE,POSITIVE,0.999728
3,The concert last night was an unforgettable ex...,POSITIVE,POSITIVE,0.999820
4,"Her presentation was clear, engaging, and well...",POSITIVE,POSITIVE,0.999872
5,I am so happy with how my new apartment turned...,POSITIVE,POSITIVE,0.999876
6,The team did an amazing job finishing the proj...,POSITIVE,POSITIVE,0.999601
7,What a wonderful sunny day for a picnic.,POSITIVE,POSITIVE,0.999887
8,The book was a delightful and heartwarming read.,POSITIVE,POSITIVE,0.999887
9,Our vacation in Bali exceeded all of our expec...,POSITIVE,POSITIVE,0.998372


In [16]:
# Run 2: harder sentences with negation, mixed sentiment, and sarcasm (10 positive, 10 negative)
sentences_run2 = [
    "The plot wasn't bad, but it definitely could've been better.",
    "I don't dislike the new update, it's actually pretty solid.",
    "Oh great, another Monday morning meeting that could've been an email.",
    "Sure, because waiting two hours for a table is exactly what I wanted.",
    "Not the worst meal I've had, but far from the best either.",
    "I wouldn't say I hated it, but I probably won't watch it again.",
    "The service wasn't great, however the food made up for it.",
    "It's hard to complain about a hotel this comfortable.",
    "I was skeptical at first, but the movie completely won me over.",
    "The instructions were confusing, yet somehow the product still works well.",
    "Yeah, losing my luggage on day one was a fantastic start to the trip.",
    "This is far from perfect, but it gets the job done.",
    "I can't say I'm impressed, but it wasn't a total disaster.",
    "The staff couldn't have been more unhelpful if they tried.",
    "It's not that the coffee was bad, it's that it was overpriced.",
    "Honestly, I expected far worse given the reviews.",
    "The app rarely crashes anymore, which is a nice change.",
    "I guess it's fine if you enjoy paying extra for nothing.",
    "Despite the rocky start, the whole experience turned out great.",
    "The design looks nice, but the battery life ruins the experience.",
]
labels_run2 = [
    "NEGATIVE", "POSITIVE", "NEGATIVE", "NEGATIVE", "NEGATIVE",
    "NEGATIVE", "POSITIVE", "POSITIVE", "POSITIVE", "POSITIVE",
    "NEGATIVE", "POSITIVE", "NEGATIVE", "NEGATIVE", "NEGATIVE",
    "POSITIVE", "POSITIVE", "NEGATIVE", "POSITIVE", "NEGATIVE",
]

df_run2 = run_sentiment_experiment("distilbert_run2_nuanced", sentences_run2, labels_run2)
df_run2

[distilbert_run2_nuanced] accuracy = 75.00%


,sentence,true_label,predicted_label,confidence
0,"The plot wasn't bad, but it definitely could'v...",NEGATIVE,NEGATIVE,0.995218
1,"I don't dislike the new update, it's actually ...",POSITIVE,POSITIVE,0.999856
2,"Oh great, another Monday morning meeting that ...",NEGATIVE,NEGATIVE,0.988933
3,"Sure, because waiting two hours for a table is...",NEGATIVE,POSITIVE,0.983871
4,"Not the worst meal I've had, but far from the ...",NEGATIVE,NEGATIVE,0.981867
5,"I wouldn't say I hated it, but I probably won'...",NEGATIVE,NEGATIVE,0.992762
6,"The service wasn't great, however the food mad...",POSITIVE,POSITIVE,0.997563
7,It's hard to complain about a hotel this comfo...,POSITIVE,POSITIVE,0.999648
8,"I was skeptical at first, but the movie comple...",POSITIVE,POSITIVE,0.999381
9,"The instructions were confusing, yet somehow t...",POSITIVE,POSITIVE,0.999620


**Observations:** Run 1 (clear-cut sentences) scored **100% accuracy** — DistilBERT is very confident (>0.998) on unambiguous positive/negative language. Run 2 (negation, mixed sentiment, sarcasm) dropped to **75% accuracy** (15/20 correct), with 5 misclassifications concentrated in sentences that mix a positive and negative clause (e.g. "Sure, because waiting two hours for a table is exactly what I wanted", "Honestly, I expected far worse given the reviews"). The model appears to key off surface-level positive words/phrases and struggles with sarcasm and double-negation, even though its confidence scores stay high (>0.98) even when wrong — showing it is confidently incorrect rather than uncertain on hard examples.


## Task 2: Named Entity Recognition

Use the pretrained `dslim/bert-base-NER` model to extract `PER` (person), `ORG` (organization),
and `LOC` (location) entities from paragraphs. Two runs of 10 paragraphs each are logged to
MLflow — one news-style batch, one biography-style batch — recording the model name, total
entity count, the distribution of entity types, and the extracted-entity table as a CSV
artifact.


In [17]:
ner_pipe = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple",
    device=-1,
)


def run_ner_experiment(run_name, paragraphs):
    rows = []
    for i, para in enumerate(paragraphs):
        for ent in ner_pipe(para):
            rows.append({
                "paragraph_id": i,
                "paragraph": para,
                "entity": ent["word"],
                "label": ent["entity_group"],
                "score": float(ent["score"]),
            })
    df = pd.DataFrame(rows)
    distribution = df["label"].value_counts().to_dict()

    artifact_path = f"{run_name}_entities.csv"
    df.to_csv(artifact_path, index=False)

    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("model_name", "dslim/bert-base-NER")
        mlflow.log_param("num_paragraphs", len(paragraphs))
        mlflow.log_metric("total_entities", len(df))
        for label, count in distribution.items():
            mlflow.log_metric(f"count_{label}", count)
        mlflow.log_artifact(artifact_path)

    os.remove(artifact_path)
    print(f"[{run_name}] total entities = {len(df)} | distribution = {distribution}")
    return df


mlflow.set_experiment("A09_Task2_Named_Entity_Recognition")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5597.77it/s]
[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<Experiment: artifact_location=('/Users/vishaniraveendran/Library/CloudStorage/GoogleDrive-vishani.20260276@iit.ac.lk/My '
 'Drive/MSC-Vish/First Year/Ai Tools and '
 'Concepts/Assignment/Assignements/A09/mlruns/2'), creation_time=1785590105281, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1785590105281, lifecycle_stage='active', name='A09_Task2_Named_Entity_Recognition', tags={}, trace_location=None, workspace='default'>

In [18]:
# Run 1: news-style paragraphs
paragraphs_run1 = [
    "Tim Cook, the CEO of Apple, unveiled the new iPhone at a press event in Cupertino, California.",
    "Serena Williams announced her retirement from professional tennis during an event hosted by ESPN in New York.",
    "The United Nations held an emergency session in Geneva to discuss the ongoing crisis in Ukraine.",
    "Elon Musk confirmed that Tesla will build a new Gigafactory near Berlin, Germany.",
    "Jeff Bezos stepped down as CEO of Amazon and was succeeded by Andy Jassy in Seattle.",
    "The World Health Organization released new guidelines from its headquarters in Geneva, Switzerland.",
    "Cristiano Ronaldo signed a new contract with Al Nassr, a football club based in Riyadh, Saudi Arabia.",
    "Google opened a new research lab in Bangalore, India, led by Sundar Pichai.",
    "The European Central Bank raised interest rates during a meeting in Frankfurt, Germany.",
    "Taylor Swift performed a sold-out concert at Wembley Stadium in London.",
]

df_ner_run1 = run_ner_experiment("bert_ner_run1_news", paragraphs_run1)
df_ner_run1

[bert_ner_run1_news] total entities = 42 | distribution = {'LOC': 19, 'PER': 11, 'ORG': 11, 'MISC': 1}


,paragraph_id,paragraph,entity,label,score
0,0,"Tim Cook, the CEO of Apple, unveiled the new i...",Tim Cook,PER,0.999788
1,0,"Tim Cook, the CEO of Apple, unveiled the new i...",Apple,ORG,0.998776
2,0,"Tim Cook, the CEO of Apple, unveiled the new i...",iPhone,MISC,0.994731
3,0,"Tim Cook, the CEO of Apple, unveiled the new i...",Cupertino,LOC,0.997840
4,0,"Tim Cook, the CEO of Apple, unveiled the new i...",California,LOC,0.999380
5,1,Serena Williams announced her retirement from ...,Serena Williams,PER,0.999558
6,1,Serena Williams announced her retirement from ...,ESPN,ORG,0.998583
7,1,Serena Williams announced her retirement from ...,New York,LOC,0.999358
8,2,The United Nations held an emergency session i...,United Nations,ORG,0.999354
9,2,The United Nations held an emergency session i...,Geneva,LOC,0.999682


In [19]:
# Run 2: biography-style paragraphs
paragraphs_run2 = [
    "Marie Curie conducted groundbreaking research on radioactivity at the University of Paris.",
    "Nelson Mandela became the first Black president of South Africa after decades of activism in Johannesburg.",
    "Albert Einstein developed the theory of relativity while working at the Swiss Patent Office in Bern.",
    "Malala Yousafzai received the Nobel Peace Prize after her advocacy work in Pakistan and at the United Nations.",
    "Barack Obama served as President of the United States and later opened the Obama Foundation in Chicago.",
    "Mahatma Gandhi led the independence movement against British rule across India.",
    "Angela Merkel served as Chancellor of Germany and represented the country at the G7 summit in Munich.",
    "Jane Goodall founded the Jane Goodall Institute after her research on chimpanzees in Tanzania.",
    "Nikola Tesla worked for Westinghouse Electric before establishing his own laboratory in New York.",
    "Kofi Annan served as Secretary-General of the United Nations, based in New York City.",
]

df_ner_run2 = run_ner_experiment("bert_ner_run2_bios", paragraphs_run2)
df_ner_run2

[bert_ner_run2_bios] total entities = 36 | distribution = {'PER': 13, 'LOC': 12, 'ORG': 7, 'MISC': 4}


,paragraph_id,paragraph,entity,label,score
0,0,Marie Curie conducted groundbreaking research ...,Marie Curie,PER,0.989940
1,0,Marie Curie conducted groundbreaking research ...,University of Paris,ORG,0.994023
2,1,Nelson Mandela became the first Black presiden...,Nelson Mandela,PER,0.999418
3,1,Nelson Mandela became the first Black presiden...,Black,MISC,0.998598
4,1,Nelson Mandela became the first Black presiden...,South Africa,LOC,0.999047
5,1,Nelson Mandela became the first Black presiden...,Johannesburg,LOC,0.999522
6,2,Albert Einstein developed the theory of relati...,Albert Einstein,PER,0.998725
7,2,Albert Einstein developed the theory of relati...,Swiss Patent Office,ORG,0.984982
8,2,Albert Einstein developed the theory of relati...,Bern,LOC,0.995165
9,3,Malala Yousafzai received the Nobel Peace Priz...,Malala Yousafzai,PER,0.974375


**Observations:** The news batch (Run 1) yielded **42 entities** (LOC 19, PER 11, ORG 11, MISC 1) while the biography batch (Run 2) yielded **36 entities** (PER 13, LOC 12, ORG 7, MISC 4) — biographies mention more people and fewer organizations than news wire text, which is skewed toward companies/institutions. The model correctly tags most well-known names, but subword tokenization causes visible errors: uncommon/compound names get split into partial fragments (`"Ma"`, `"##hat"`, `"##ma Gandhi"`; `"Ko"`, `"##fi Annan"`; `"C"`, `"##rist"`, `"##iano Ronaldo"`), and one clear mislabel occurs where `"Elon Musk"` is tagged `ORG` and `"Tesla"` is tagged `PER` (the two labels are swapped) with much lower confidence (0.95 and 0.86) than the typical >0.99 seen on correctly tagged entities — low confidence is a useful signal for flagging likely NER mistakes.


## Task 3: Text Generation

Use the pretrained `gpt2` model to generate text continuations for five prompts. Two runs are
logged to MLflow with different sampling parameters (`temperature`, `max_length`) so the effect
of each parameter on the generated text can be compared; each run logs its parameters, the
generated outputs as a text artifact, and a short observations file as a second artifact.


In [20]:
generator = pipeline("text-generation", model="gpt2", device=-1)

prompts = [
    "The future of artificial intelligence is",
    "In a small village by the mountains,",
    "The best way to learn a new skill is",
    "Scientists recently discovered that",
    "On a rainy afternoon, she decided to",
]


def run_generation_experiment(run_name, temperature, max_length, seed=42):
    set_seed(seed)
    outputs = []
    for prompt in prompts:
        result = generator(
            prompt,
            max_length=max_length,
            max_new_tokens=None,  # avoid pipeline default overriding max_length
            temperature=temperature,
            do_sample=True,
            num_return_sequences=1,
            truncation=True,
            pad_token_id=generator.tokenizer.eos_token_id,
        )
        outputs.append(result[0]["generated_text"])

    output_text = "\n\n".join(
        f"Prompt: {p}\nGenerated: {o}" for p, o in zip(prompts, outputs)
    )
    artifact_path = f"{run_name}_generations.txt"
    with open(artifact_path, "w") as f:
        f.write(output_text)

    avg_len = sum(len(o) for o in outputs) / len(outputs)
    observations = (
        f"Run: {run_name}\n"
        f"temperature={temperature}, max_length={max_length}\n"
        f"Average generated length (characters): {avg_len:.1f}\n"
    )
    obs_path = f"{run_name}_observations.txt"
    with open(obs_path, "w") as f:
        f.write(observations)

    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("model_name", "gpt2")
        mlflow.log_param("temperature", temperature)
        mlflow.log_param("max_length", max_length)
        mlflow.log_param("num_prompts", len(prompts))
        mlflow.log_metric("avg_output_length_chars", avg_len)
        mlflow.log_artifact(artifact_path)
        mlflow.log_artifact(obs_path)

    os.remove(artifact_path)
    os.remove(obs_path)

    for p, o in zip(prompts, outputs):
        print(f"PROMPT: {p}\nGENERATED: {o}\n")
    return outputs


mlflow.set_experiment("A09_Task3_Text_Generation")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7405.92it/s]


<Experiment: artifact_location=('/Users/vishaniraveendran/Library/CloudStorage/GoogleDrive-vishani.20260276@iit.ac.lk/My '
 'Drive/MSC-Vish/First Year/Ai Tools and '
 'Concepts/Assignment/Assignements/A09/mlruns/3'), creation_time=1785590158956, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1785590158956, lifecycle_stage='active', name='A09_Task3_Text_Generation', tags={}, trace_location=None, workspace='default'>

In [21]:
# Run 1: low temperature, shorter output -> more focused/conservative text
outputs_run1 = run_generation_experiment("gpt2_run1_low_temp", temperature=0.7, max_length=50)

PROMPT: The future of artificial intelligence is
GENERATED: The future of artificial intelligence is uncertain, but a new report in the journal Proceedings of the National Academy of Sciences shows that we can learn from our mistakes.

The study, led by neuroscientist Alan Gershman at Queen's University

PROMPT: In a small village by the mountains,
GENERATED: In a small village by the mountains, a family has gathered in their summer camp to celebrate. At the end of the day, the locals are the only people in their community. But the village is not the only one. The people of the village

PROMPT: The best way to learn a new skill is
GENERATED: The best way to learn a new skill is to learn what you already know. That means getting a sense of what others are thinking about you and what they think you're doing.

You may have read articles about how to communicate your message,

PROMPT: Scientists recently discovered that
GENERATED: Scientists recently discovered that the same bacteria can g

In [22]:
# Run 2: high temperature, longer output -> more random/creative text
outputs_run2 = run_generation_experiment("gpt2_run2_high_temp", temperature=1.3, max_length=100)

PROMPT: The future of artificial intelligence is
GENERATED: The future of artificial intelligence is uncertain and could ultimately create massive technological gaps."

PROMPT: In a small village by the mountains,
GENERATED: In a small village by the mountains, they are already moving toward a new city...

There was a great deal of discontent in the capital and its rural population. The town had been ravaged when Hohin, a leader in his father's administration after the Battle of Endless Sea, was killed with his party by a strong party loyal to Hohin. The town in which Hohin lived belonged to the nobility who became its new rulers, while the town had belonged to others,

PROMPT: The best way to learn a new skill is
GENERATED: The best way to learn a new skill is to learn it while trying to stay focused. Learn it, if you like, as many times as you possibly can during school."

The goal for learning new skills is to improve your overall performance, which typically isn't very often for a 

**Observations:** Run 1 (temperature=0.7, max*length=50) produces short, coherent, on-topic continuations that stay grounded in the prompt (average ~230 characters). Run 2 (temperature=1.3, max_length=100) produces noticeably longer (~380 characters) and more diverse/creative text, but coherence degrades — it drifts topic more freely, invents unrelated details, and occasionally produces disjointed or nonsensical passages. This confirms the expected trade-off: lower temperature favors focused, predictable text, while higher temperature favors variety and creativity at the cost of factual/logical consistency. *(Note: the pipeline call explicitly passes `max_new_tokens=None` alongside `max_length` — without this, newer versions of `transformers` silently let a default `max_new_tokens` override `max_length`, which caused the two runs to ignore the intended length setting.)\_


## Viewing the MLflow Experiments

All six runs (two per task) are stored in `./mlflow.db` (SQLite) in this folder. To inspect them
in the MLflow UI, run the following from the `A09` directory and open the printed URL in a
browser:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

Each experiment (`A09_Task1_Sentiment_Classification`, `A09_Task2_Named_Entity_Recognition`,
`A09_Task3_Text_Generation`) contains two runs that can be compared side by side using the
parameters and metrics logged above.
